# 0050 vs 009816 research
Run All after editing USER CONFIG. Add FINLAB_API_TOKEN and TIINGO_API_TOKEN in Colab Secrets. FX excluded. Approve Drive mount.
Only real post-launch ETF history; no hypothetical reconstruction. Short annualized results need caution.


In [ ]:
# USER CONFIG — only change this block for normal use
ASSETS = ['TW:0050', 'TW:009816', 'TW:0056', 'US:VOO', 'US:VGT']
FX_MODE = "none"
PORTFOLIOS = {'TW 0050 only': {'TW:0050': 1.0}, '0050 + VOO': {'TW:0050': 0.33, 'US:VOO': 0.67}, '0050 + VOO + VGT': {'TW:0050': 0.3, 'US:VOO': 0.5, 'US:VGT': 0.2}, '009816 + VOO + VGT': {'TW:009816': 0.3, 'US:VOO': 0.5, 'US:VGT': 0.2}}
BENCHMARK = "TW:0050"
DATE_MODE = "intersection"
CUSTOM_START_DATE = None
CUSTOM_END_DATE = None
STRICT = False
RETURN_TYPE = "total_return"
RISK_FREE_RATE = 0.0
EXPORT_TO_DRIVE = True
DRIVE_OUTPUT_ROOT = "/content/drive/MyDrive/00Quant_Research/compare_stock_return"
USE_CACHE = True
REFRESH_CACHE = False
DISPLAY_ASSETS = None
FEES = {}
CUSTOM_ASSETS = {}
HOLDINGS_FILES = {}
NON_TRADING_DATES = {"TW:0050": {"2025-06-11": "market_suspension", "2025-06-12": "market_suspension", "2025-06-13": "market_suspension", "2025-06-16": "market_suspension", "2025-06-17": "market_suspension"}}


## Install → import → Drive → authentication → environment


In [ ]:
from pathlib import Path
import importlib, subprocess, sys, os

# Use a dedicated checkout path so the repository directory itself cannot shadow
# the installed src-layout package in Colab. Re-running this cell updates the checkout.
REPO = Path("/content/compare_stock_return_repo")
if (REPO / ".git").is_dir():
    subprocess.run(["git", "-C", str(REPO), "pull", "--ff-only", "origin", "main"], check=True)
else:
    if REPO.exists():
        raise RuntimeError(f"{REPO} exists but is not a git checkout; remove it or restart the runtime")
    subprocess.run(["git", "clone", "https://github.com/hh4832/compare_stock_return.git", str(REPO)], check=True)

os.chdir(REPO)
sys.path.insert(0, str(REPO / "src"))
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", "requirements.txt"], check=True)
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-e", "."], check=True)

# Prefer this checkout's src tree explicitly. Also clear stale modules when a Colab
# cell is re-run after updating the repository.
SRC = str(REPO / "src")
if SRC in sys.path:
    sys.path.remove(SRC)
sys.path.insert(0, SRC)
for _name in list(sys.modules):
    if _name == "compare_stock_return" or _name.startswith("compare_stock_return."):
        del sys.modules[_name]
importlib.invalidate_caches()

from compare_stock_return.config import Config
from compare_stock_return.pipeline import run
from compare_stock_return.data_loader import authenticate
from compare_stock_return.reporting import environment

# Fail early if Python resolved a different package with the same name.
import compare_stock_return.config as _config_module
_resolved = Path(_config_module.__file__).resolve()
_expected_root = (REPO / "src").resolve()
if _expected_root not in _resolved.parents:
    raise ImportError(f"compare_stock_return resolved from unexpected location: {_resolved}")
print("Package source:", _resolved)

from google.colab import drive
drive.mount("/content/drive")
if not EXPORT_TO_DRIVE:
    raise ValueError("This Colab notebook requires EXPORT_TO_DRIVE=True; use CLI for local output")
if not Path(DRIVE_OUTPUT_ROOT).is_dir():
    raise FileNotFoundError(f"Create this exact Google Drive folder first: {DRIVE_OUTPUT_ROOT}")
print("Output root:", DRIVE_OUTPUT_ROOT)
if any(a.startswith("TW:") for a in ASSETS):
    authenticate()
if any(a.startswith("US:") for a in ASSETS):
    from compare_stock_return.providers.tiingo_provider import token_from_environment
    token_from_environment()  # no display or serialization
print("Configured providers:", sorted({a.split(":")[0] for a in ASSETS}))
print(environment())


## Shared engine and common-period headline results


In [ ]:
config = Config(assets=ASSETS, fx_mode=FX_MODE, portfolios=PORTFOLIOS, benchmark=BENCHMARK, date_mode=DATE_MODE,
    custom_start_date=CUSTOM_START_DATE, custom_end_date=CUSTOM_END_DATE, strict=STRICT,
    return_type=RETURN_TYPE, risk_free_rate=RISK_FREE_RATE, drive_output_root=DRIVE_OUTPUT_ROOT,
    use_cache=USE_CACHE, refresh_cache=REFRESH_CACHE, display_symbols=DISPLAY_ASSETS,
    fees=FEES, custom_assets=CUSTOM_ASSETS, holdings_files=HOLDINGS_FILES,
    non_trading_dates=NON_TRADING_DATES)
result = run(config)
from IPython.display import display, HTML
import pandas as pd
display(result["intersection_metrics"])
display(result["model_portfolio_metrics"])
display(pd.read_csv(result["run_path"] / "results/assets.csv"))
print("Full-history metrics are supplementary and use different date ranges.")
charts = result["run_path"] / "charts"
for name in ["normalized_equity_curve","underwater","rolling_12m_return","rolling_36m_return","rolling_36m_volatility","rolling_sharpe","rolling_excess_return","model_portfolio_equity_curve","model_portfolio_drawdown","monthly_return_heatmap"]:
    display(HTML((charts / f"{name}.html").read_text()))


## Benchmark, pairwise, drawdowns, fees and optional holdings
Holdings require official dated snapshots. Fee metadata is not an extra deduction. No linkage-fund NAV is assumed.


In [ ]:
for name in ["benchmark_metrics", "pairwise_metrics", "drawdowns", "monthly_excess_returns", "fees", "holdings_concentration", "holdings_overlap"]:
    path = result["run_path"] / "results" / f"{name}.csv"
    if path.exists() and path.stat().st_size > 1:
        print(name)
        display(pd.read_csv(path))
    else:
        print(name, "skipped / unavailable")
print(result["metadata"]["holdings_status"])
